In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
src_paths = {
    "soil": "/content/drive/MyDrive/Chatbot/SoilClassificationDataset/CyAUG-Dataset/",
    "disease": "/content/drive/MyDrive/Chatbot/PlantVillage/plantvillage dataset/color/",
    "weed": "/content/drive/MyDrive/Chatbot/DeepWeeds/images/",
    "pest": "/content/drive/MyDrive/Chatbot/Pestopia/Datasets/Pest_Dataset/",
    "crop": "/content/drive/MyDrive/Chatbot/Crop Identification/Agricultural-crops/"
}
dest_path="/content/drive/MyDrive/Chatbot/Routing_model_dataset/"

In [ ]:
# Soil

import os
import random
import shutil

TARGET = 400

soil_src = src_paths["soil"]
soil_dest = os.path.join(dest_path, "soil")

os.makedirs(soil_dest, exist_ok=True)

classes = os.listdir(soil_src)

for cls in classes:
    src_cls_path = os.path.join(soil_src, cls)

    images = [img for img in os.listdir(src_cls_path)
              if os.path.isfile(os.path.join(src_cls_path, img))]

    if len(images) > TARGET:
        selected = random.sample(images, TARGET)
    else:
        selected = images

    for img in selected:
        src_img = os.path.join(src_cls_path, img)

        # rename to avoid collisions
        new_name = f"{cls}_{img}"
        dst_img = os.path.join(soil_dest, new_name)

        if not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)

    print(f"{cls}: {len(selected)} images added")

Alluvial_Soil: 400 images added
Arid_Soil: 284 images added
Black_Soil: 400 images added
Laterite_Soil: 219 images added
Mountain_Soil: 201 images added
Red_Soil: 400 images added
Yellow_Soil: 400 images added


In [ ]:
# Plant - Disease

import os
import random
import shutil

TARGET_TOTAL = 3000

disease_src = src_paths["disease"]
disease_dest = os.path.join(dest_path, "disease")

os.makedirs(disease_dest, exist_ok=True)

classes = os.listdir(disease_src)
num_classes = len(classes)

per_class = TARGET_TOTAL // num_classes

print(f"Classes: {num_classes}, Per class: {per_class}")

for cls in classes:
    src_cls_path = os.path.join(disease_src, cls)

    images = [img for img in os.listdir(src_cls_path)
              if os.path.isfile(os.path.join(src_cls_path, img))]

    if len(images) > per_class:
        selected = random.sample(images, per_class)
    else:
        selected = images  # take all if less

    for img in selected:
        src_img = os.path.join(src_cls_path, img)
        new_name = f"{cls}_{img}"
        dst_img = os.path.join(disease_dest, new_name)

        if not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)

    print(f"{cls}: {len(selected)} images added")

Classes: 38, Per class: 78
Apple___Apple_scab: 78 images added
Apple___Black_rot: 78 images added
Apple___Cedar_apple_rust: 78 images added
Apple___healthy: 78 images added
Blueberry___healthy: 78 images added
Cherry_(including_sour)___Powdery_mildew: 78 images added
Cherry_(including_sour)___healthy: 78 images added
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 78 images added
Corn_(maize)___Common_rust_: 78 images added
Corn_(maize)___Northern_Leaf_Blight: 78 images added
Corn_(maize)___healthy: 78 images added
Grape___Black_rot: 78 images added
Grape___Esca_(Black_Measles): 78 images added
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 78 images added
Grape___healthy: 78 images added
Orange___Haunglongbing_(Citrus_greening): 78 images added
Peach___Bacterial_spot: 78 images added
Peach___healthy: 78 images added
Pepper,_bell___Bacterial_spot: 78 images added
Pepper,_bell___healthy: 78 images added
Potato___Early_blight: 78 images added
Potato___Late_blight: 78 images added
Potato

In [ ]:
import os
import shutil
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img

TARGET_TOTAL = 3000

selected_classes = [
    "large cutworm",
    "Locust",
    "Pieris canidia",
    "red spider",
    "yellow rice borer"
]

pest_src = src_paths["pest"]
pest_dest = os.path.join(dest_path, "pest")

os.makedirs(pest_dest, exist_ok=True)

all_images = []

# -------- STEP 1: COPY ALL ORIGINAL IMAGES (FLAT) --------
for cls in selected_classes:
    src_cls_path = os.path.join(pest_src, cls)

    images = [img for img in os.listdir(src_cls_path)
              if os.path.isfile(os.path.join(src_cls_path, img))]

    for img in images:
        src_img = os.path.join(src_cls_path, img)

        # rename to avoid conflict (NO subfolders)
        new_name = f"{cls.replace(' ', '_')}_{img}"
        dst_img = os.path.join(pest_dest, new_name)

        if not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)

        all_images.append(new_name)

print(f"Original images copied: {len(all_images)}")

# -------- STEP 2: AUGMENT --------
current_count = len(os.listdir(pest_dest))
needed = TARGET_TOTAL - current_count

print(f"Need to generate: {needed}")

datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    shear_range=0.15
)

images = os.listdir(pest_dest)

i = 0
while needed > 0:
    img_name = images[i % len(images)]
    img_path = os.path.join(pest_dest, img_name)

    try:
        img = load_img(img_path)
        x = img_to_array(img)
        x = x.reshape((1,) + x.shape)

        for batch in datagen.flow(x, batch_size=1):
            new_img = array_to_img(batch[0])

            # add "_aug" to avoid conflict
            name, ext = os.path.splitext(img_name)
            new_name = f"{name}_aug_{needed}{ext}"

            new_img.save(os.path.join(pest_dest, new_name))
            needed -= 1
            break

    except:
        pass

    i += 1

print(f"Final count: {len(os.listdir(pest_dest))}")

Original images copied: 1525
Need to generate: 715
Final count: 3000


In [ ]:
import os
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img

TARGET_TOTAL = 3000

crop_src = src_paths["crop"]
crop_dest = os.path.join(dest_path, "crop")

os.makedirs(crop_dest, exist_ok=True)

# -------- STEP 1: COPY ORIGINALS (FLAT + RENAME) --------
all_images = []

for cls in os.listdir(crop_src):
    src_cls_path = os.path.join(crop_src, cls)

    if not os.path.isdir(src_cls_path):
        continue

    images = [img for img in os.listdir(src_cls_path)
              if os.path.isfile(os.path.join(src_cls_path, img))]

    for img in images:
        src_img = os.path.join(src_cls_path, img)

        new_name = f"{cls.replace(' ', '_')}_{img}"
        dst_img = os.path.join(crop_dest, new_name)

        if not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)

        all_images.append(new_name)

print(f"Original images: {len(all_images)}")

# -------- STEP 2: AUGMENT (MIXED, NO SEPARATION) --------
current_count = len(os.listdir(crop_dest))
needed = TARGET_TOTAL - current_count

print(f"Need to generate: {needed}")

datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    shear_range=0.15
)

images = os.listdir(crop_dest)

aug_counter = 1
i = 0

while needed > 0:
    img_name = images[i % len(images)]
    img_path = os.path.join(crop_dest, img_name)

    try:
        img = load_img(img_path)
        x = img_to_array(img)
        x = x.reshape((1,) + x.shape)

        for batch in datagen.flow(x, batch_size=1):
            new_img = array_to_img(batch[0])

            name, ext = os.path.splitext(img_name)
            new_name = f"{name}_aug{aug_counter}{ext}"

            new_img.save(os.path.join(crop_dest, new_name))

            aug_counter += 1
            needed -= 1
            break

    except:
        pass

    i += 1

print(f"Final count: {len(os.listdir(crop_dest))}")

Original images: 829
Need to generate: 2171
Final count: 3000


In [ ]:
import os
import random
import shutil
import re

TARGET_TOTAL = 3000

weed_src = src_paths["weed"]
weed_dest = os.path.join(dest_path, "weed")

os.makedirs(weed_dest, exist_ok=True)

# -------- STEP 1: GROUP IMAGES BY LABEL (0–8) --------
label_map = {str(i): [] for i in range(9)}

for img in os.listdir(weed_src):
    if not img.endswith(".jpg"):
        continue

    # extract label (digit before .jpg)
    match = re.search(r'-(\d)\.jpg$', img)
    if match:
        label = match.group(1)
        label_map[label].append(img)

# -------- STEP 2: SAMPLE EQUALLY --------
num_classes = 9
per_class = TARGET_TOTAL // num_classes  # ~333

print(f"Per class target: {per_class}")

total_selected = 0

for label in label_map:
    images = label_map[label]

    if len(images) > per_class:
        selected = random.sample(images, per_class)
    else:
        selected = images

    for img in selected:
        src_img = os.path.join(weed_src, img)

        # rename to avoid conflicts (optional but safe)
        new_name = f"weed{label}_{img}"
        dst_img = os.path.join(weed_dest, new_name)

        if not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)

    print(f"Label {label}: {len(selected)} images")
    total_selected += len(selected)

print(f"Total images copied: {total_selected}")

Per class target: 333
Label 0: 333 images
Label 1: 333 images
Label 2: 333 images
Label 3: 333 images
Label 4: 27 images
Label 5: 0 images
Label 6: 0 images
Label 7: 0 images
Label 8: 0 images
Total images copied: 1359


In [3]:
!pip install icrawler

In [4]:
from icrawler.builtin import BingImageCrawler
import os

save_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset/ocr_dataset"

queries = [
    # fertilizers
    "fertilizer bag label india",
    "urea fertilizer bag instructions",
    "npk fertilizer label dosage",
    "dap fertilizer bag label",
    "organic fertilizer packaging label",
    "micronutrient fertilizer label",
    "liquid fertilizer bottle instructions",
    "fertilizer composition label",

    # pesticides
    "pesticide bottle label instructions",
    "insecticide label dosage agriculture",
    "fungicide packet label instructions",
    "herbicide label usage instructions",
    "pesticide safety instructions label",
    "crop spray chemical bottle label",
    "agriculture pesticide india label",

    # seeds
    "seed packet label agriculture",
    "hybrid seeds packet label",
    "vegetable seeds packet back side info",
    "crop seed instructions label",
    "seed germination instructions packet",

    # bills
    "fertilizer bill invoice india",
    "agriculture shop receipt india",
    "pesticide purchase bill",
    "seed shop invoice agriculture",
    "agri input bill receipt",

    # soil reports
    "soil test report india",
    "soil health card india sample",
    "agriculture lab report soil test",
    "soil nutrient report sample",

    # forms
    "pm kisan form document",
    "farmer subsidy form india",
    "agriculture application form filled",
    "crop insurance form india",
    "farmer registration document india",

    # misc
    "crop advisory poster agriculture",
    "agriculture instruction board farm",
    "irrigation instructions poster",
    "fertilizer recommendation chart",
    "pesticide spray schedule chart"
]

for q in queries:
    folder = os.path.join(save_path, q.replace(" ", "_"))
    os.makedirs(folder, exist_ok=True)

    crawler = BingImageCrawler(storage={"root_dir": folder})
    crawler.crawl(keyword=q, max_num=100)

ERROR:downloader:Response status code 404, file https://www.wikihow.com/images/thumb/2/22/Read-a-Fertilizer-Label-Step-1-Version-2.jpg
ERROR:downloader:Response status code 403, file https://assetcloud01.roccommerce.net/w1500-h1500-cpad/_reinders/4/5/6/rp19-0-0d.jpg
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: "X-Content-Type-Options : nosniff\r\nContent-Security-Policy: default-src 'self'; connect-src *; font-src *; frame-src *; img-src * data:; media-src *; object-src *; script-src * 'unsafe-inline' 'unsafe-eval'; style-src * 'unsafe-inline';\r\nX-Powered-By: MSME\r\nReferrer-Poli

In [5]:
import os
import shutil

base_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset/ocr_dataset"

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            src = os.path.join(folder_path, file)

            if os.path.isfile(src):
                # new name = foldername_filename
                new_name = f"{folder}_{file}"
                dst = os.path.join(base_path, new_name)

                # avoid overwrite if duplicate again
                count = 1
                while os.path.exists(dst):
                    name, ext = os.path.splitext(new_name)
                    dst = os.path.join(base_path, f"{name}_{count}{ext}")
                    count += 1

                shutil.move(src, dst)

print("Done")

Done


In [9]:
import os

folder = "/content/drive/MyDrive/Chatbot/Routing_model_dataset/ocr_dataset"

total_files = 0
total_size = 0  # in bytes

for file in os.listdir(folder):
    path = os.path.join(folder, file)

    if os.path.isfile(path):
        total_files += 1
        total_size += os.path.getsize(path)

# convert size to MB
total_size_mb = total_size / (1024 * 1024)

print("Total files:", total_files)
print("Total size: {:.2f} MB".format(total_size_mb))

Total files: 2853
Total size: 467.47 MB


In [8]:
from icrawler.builtin import BingImageCrawler

save_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset/ocr_dataset"

new_queries = [
    "fertilizer bag backside instructions india",
    "npk fertilizer label composition chart",
    "bio fertilizer packet label india",
    "insecticide bottle back label instructions",
    "fungicide label dosage crop specific",
    "herbicide mixing instructions label",

    "rice seed packet label india",
    "wheat seed packet instructions label",
    "cotton seed bag label india",
    "vegetable seed packet backside details",
    "hybrid seed packet farming label",

    "fertilizer shop bill handwritten india",
    "agriculture store invoice printed india",
    "pesticide bill receipt handwritten",
    "seed shop bill handwritten india",
    "agri purchase receipt rural india",

    "soil health card report real india",
    "soil lab report handwritten values",
    "farmer land record document india",
    "crop advisory printed sheet agriculture",
    "agriculture extension pamphlet india",

    "pm kisan registration form filled sample",
    "farmer loan application form india filled",
    "crop insurance claim form filled india",
    "land ownership document india scan",
    "farmer id card india document"
]

for q in new_queries:
    crawler = BingImageCrawler(storage={"root_dir": save_path})
    crawler.crawl(
        keyword=q,
        max_num=100,
        file_idx_offset="auto"
    )

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: "X-Content-Type-Options : nosniff\r\nContent-Security-Policy: default-src 'self'; connect-src *; font-src *; frame-src *; img-src * data:; media-src *; object-src *; script-src * 'unsafe-inline' 'unsafe-eval'; style-src * 'unsafe-inline';\r\nX-Powered-By: MSME\r\nReferrer-Policy: same-origin\r\nDate: Thu, 19 Mar 2026 13:48:42 GMT\r\nContent-Length: 527594\r\n\r\n"
ERROR:downloader:Exception caught when downloading file https://www.msmemart.com/upload/products/1594303126126.jpg, error: HTTPSConnectionPool(host='www.msmemart.com', port=443)